# Emre'nin sesiyle Türkçe Piper (VITS) eğitimi

Veri üretim notebook'unun çıktısı (`emre_tr`) ile Piper'ın **LJSpeech medium** checkpoint'inden ince ayar yapar.
Başlangıç modeli kamu malı veriyle sıfırdan eğitildiği için lisans temiz. Metni espeak yerine kendi Türkçe G2P'mizle
fonemlere çeviriyoruz; çıkan ONNX modeli telefonda espeak olmadan çalışır.

**Kaggle'da bir kez yapılacaklar**
1. *Settings → Accelerator → GPU T4 x2* (veya P100), *Internet → On*.
2. *Add Input → Your Work* sekmesinden veri üretim notebook'unu seç (çıktısı `emre_tr` klasörü ve `emre_tr.zip`).
3. *Save Version → Save & Run All (Commit)*. Eğitim `EGITIM_SURESI` kadar sürer, sonunda ONNX modeli ve örnek sesler
   *Output* sekmesinde olur.
4. Devam eğitimi için: bu notebook'un önceki çıktısını da *Add Input* ile ekle; en son `.ckpt` otomatik bulunur.

In [ ]:
# --- Ayarlar ---
EGITIM_SURESI = "00:09:30:00"   # gün:saat:dakika:saniye; Kaggle oturumu 12 saatte kapanır, kurulum ve dışa aktarma için pay bırak
BATCH_SIZE    = 16
KAYIT_ARALIGI = 5               # kaç epoch'ta bir checkpoint
TEST_CUMLELERI = [
    "Merhaba, ben Emre. Bu ses tamamen telefonda üretiliyor.",
    "Bin dokuz yüz elli üç yılında kurulan şirket, üçüncü kuşakla yüzde kırk büyüdü.",
    "Kitabın ikinci bölümünde yazar, Ankara'daki çocukluk yıllarını anlatıyor.",
    "Toplantı yarın sabah saat onda mı başlayacak?",
]
import glob, os, subprocess, sys, json, shutil
CALISMA = os.environ.get("CALISMA_DIZINI", "/kaggle/working")
GIRDI   = os.environ.get("GIRDI_DIZINI", "/kaggle/input")
ARAC    = os.environ.get("ARAC_DIZINI", "/tmp/egitim")
os.makedirs(ARAC, exist_ok=True)

In [ ]:
# --- Ortam: Piper eğitim kodu eski bir PyTorch Lightning istiyor, bu yüzden ayrı bir Python 3.11 ortamı kuruyoruz (5-8 dk) ---
VENV = f"{ARAC}/venv"
PY = f"{VENV}/bin/python"
def sh(komut):
    print("$", komut[:120]); r = subprocess.run(komut, shell=True, capture_output=True, text=True)
    if r.returncode: print(r.stdout[-2000:], r.stderr[-3000:]); raise RuntimeError("komut başarısız")
    return r.stdout
if not os.path.exists(PY):
    sh("pip install -q uv")
    sh(f"uv venv -q -p 3.11 {VENV}")
    sh(f"uv pip install -q -p {PY} torch==2.1.2 pytorch-lightning==1.9.5 'numpy<2' librosa==0.10.2 cython onnx onnxruntime soundfile 'setuptools<70'")
PIPER = f"{ARAC}/piper"
if not os.path.exists(PIPER):
    sh(f"git clone -q https://github.com/rhasspy/piper.git {PIPER} && cd {PIPER} && git checkout -q 73c04d81d5590ecc46e522de3601ce7fb29fc2be")
    sh(f"cd {PIPER}/src/python/piper_train/vits/monotonic_align && mkdir -p monotonic_align && {VENV}/bin/cythonize -q -i core.pyx && mv core*.so monotonic_align/")
    # Piper tüm callback'leri ModelCheckpoint ile değiştiriyor; --max_time'ın Timer'ı da siliniyordu. Sadece eski checkpoint callback'ini değiştir.
    ana = f"{PIPER}/src/python/piper_train/__main__.py"
    k = open(ana).read()
    eski = "trainer.callbacks = [ModelCheckpoint(every_n_epochs=args.checkpoint_epochs)]"
    assert eski in k, "piper __main__ beklenen satırı içermiyor"
    open(ana, "w").write(k.replace(eski, "trainer.callbacks = [c for c in trainer.callbacks if not isinstance(c, ModelCheckpoint)] + [ModelCheckpoint(every_n_epochs=args.checkpoint_epochs)]"))
PIPER_PY = f"{PIPER}/src/python"
print(sh(f"{PY} -c 'import torch, pytorch_lightning as pl; print(torch.__version__, pl.__version__, torch.cuda.device_count(), \"GPU\")'"))

In [ ]:
G2P_KODU = r'''"""Kural tabanlı Türkçe G2P (harf → IPA), espeak-ng kullanmadan.

Çıktı, Piper'ın espeak sembol kümesindeki sembollerle yazılır; böylece Piper/VITS
checkpoint'lerinin `phoneme_id_map`'i olduğu gibi kullanılabilir ama çalışma anında
GPL'li espeak'e ihtiyaç kalmaz.

Kapsam (v0):
- Türkçe büyük/küçük harf (İ/ı) doğru dönüşüm, şapkalı harfler (â, î, û).
- k, g, l için ince/kalın ünlü bağlamına göre allofonlar (c, ɟ, l / k, ɡ, ɫ).
- ğ: ince ünlüler arasında j, diğer durumlarda önceki ünlüyü uzatır.
- Vurgu: varsayılan son hece; soru eki ve bağlaç gibi vurgusuz kelimeler; küçük bir
  istisna sözlüğü (yer adları, bazı zarflar); kesme işaretinden sonraki ekler vurgusuz.
Sayı/tarih/kısaltma açma bu modülün işi değil; metin önce normalize edilmeli.
"""
import re
from typing import Dict, List, Optional

INCE = set("eiöüîê")
KALIN = set("aıouâû")
UNLULER = INCE | KALIN

_HARF = {
    "a": "a", "b": "b", "c": "dʒ", "ç": "tʃ", "d": "d", "e": "e", "f": "f",
    "h": "h", "ı": "ɯ", "i": "i", "j": "ʒ", "m": "m", "n": "n", "o": "o",
    "ö": "ø", "p": "p", "r": "ɾ", "s": "s", "ş": "ʃ", "t": "t", "u": "u",
    "ü": "y", "v": "v", "y": "j", "z": "z", "q": "k", "w": "v", "x": "ks",
    "â": "aː", "î": "iː", "û": "uː", "ê": "eː",
}

# Vurgusuz (klitik) kelimeler: vurgu bir önceki kelimede kalır.
VURGUSUZ = {"mı", "mi", "mu", "mü", "mısın", "misin", "musun", "müsün", "mıyım", "miyim",
            "muyum", "müyüm", "mıdır", "midir", "mudur", "müdür", "de", "da", "ki", "ve", "ile"}

# Vurgusu son hecede olmayan kelimeler: kelime → vurgulu ünlünün sırası (0'dan).
VURGU_ISTISNA: Dict[str, int] = {
    "ankara": 0, "istanbul": 1, "izmir": 0, "antalya": 1, "türkiye": 0, "avrupa": 1,
    "almanya": 1, "amerika": 1, "londra": 0, "paris": 0, "adana": 0, "bursa": 0,
    "konya": 0, "trabzon": 0, "kayseri": 0, "samsun": 0, "erzurum": 0, "eskişehir": 0,
    "şimdi": 0, "sonra": 0, "önce": 0, "belki": 0, "yalnız": 0, "ancak": 0, "nasıl": 0,
    "nerede": 0, "nereye": 0, "neden": 0, "niçin": 0, "hangi": 0, "bazen": 0, "çünkü": 0,
    "yarın": 0, "dün": 0, "bugün": 0, "akşam": 0, "sabah": 0, "elbette": 1, "lütfen": 0,
    "evet": 0, "hayır": 0, "merhaba": 0, "tamam": 1, "efendim": 0, "işte": 0, "hemen": 0,
    "aslında": 0, "galiba": 1, "gerçekten": 0, "maalesef": 1, "örneğin": 0, "yani": 0,
    "fakat": 0, "lakin": 0, "ama": 0, "oysa": 0, "sanki": 0, "hatta": 0, "henüz": 0,
}

NOKTALAMA = {",": ",", ".": ".", "?": "?", "!": "!", ";": ";", ":": ":",
             "…": ".", "-": ",", "–": ",", "—": ",", "(": ",", ")": ","}


def kucult(metin: str) -> str:
    return metin.replace("I", "ı").replace("İ", "i").lower()


def _baglam_ince(kelime: str, i: int) -> bool:
    """k/g/l için: aynı hecedeki ünlü ince mi? Önce sonraki, yoksa önceki ünlüye bakar."""
    if i + 1 < len(kelime) and kelime[i + 1] in UNLULER:
        return kelime[i + 1] in INCE
    for j in range(i - 1, -1, -1):
        if kelime[j] in UNLULER:
            return kelime[j] in INCE
    for j in range(i + 1, len(kelime)):
        if kelime[j] in UNLULER:
            return kelime[j] in INCE
    return False


def _kelime_fonemleri(kelime: str) -> List[List[str]]:
    """Her harf için fonem listesi döndürür (ğ gibi sessiz harfler boş liste)."""
    sonuc: List[List[str]] = []
    for i, h in enumerate(kelime):
        if h in "kg":
            ince = _baglam_ince(kelime, i)
            # Şapkalı a (kâr, gâvur) önündeki k/g incelir
            if i + 1 < len(kelime) and kelime[i + 1] == "â":
                ince = True
            sonuc.append([("c" if h == "k" else "ɟ") if ince else ("k" if h == "k" else "ɡ")])
        elif h == "l":
            ince = _baglam_ince(kelime, i) or (i + 1 < len(kelime) and kelime[i + 1] == "â")
            sonuc.append(["l" if ince else "ɫ"])
        elif h == "ğ":
            onceki = kelime[i - 1] if i else ""
            sonraki = kelime[i + 1] if i + 1 < len(kelime) else ""
            if onceki in INCE and sonraki in UNLULER:
                sonuc.append(["j"])
            elif onceki in UNLULER and sonraki not in UNLULER and sonuc and "ː" not in sonuc[-1][-1]:
                sonuc[-1] = sonuc[-1][:-1] + [sonuc[-1][-1] + "ː"]
                sonuc.append([])
            else:
                sonuc.append([])
        elif h in _HARF:
            sonuc.append(list(_HARF[h]) if h in "cçx" else [_HARF[h]])
        else:
            sonuc.append([])
    return sonuc


def _vurgulu_unlu(kelime: str, kok_uzunlugu: int) -> Optional[int]:
    """Vurgulu ünlünün kelime içindeki harf indeksini döndürür."""
    unluler = [i for i, h in enumerate(kelime) if h in UNLULER]
    if not unluler:
        return None
    kok = kelime[:kok_uzunlugu]
    kok_unluler = [i for i in unluler if i < kok_uzunlugu]
    if kok in VURGU_ISTISNA and kok_unluler:
        return kok_unluler[min(VURGU_ISTISNA[kok], len(kok_unluler) - 1)]
    if kok_uzunlugu < len(kelime) and kok_unluler:  # kesme işaretinden sonraki ekler vurgusuz
        return kok_unluler[-1]
    return unluler[-1]


def _kelime_ipa(ham: str) -> List[str]:
    kok_uzunlugu = len(ham)
    if "'" in ham:
        kok_uzunlugu = ham.index("'")
        ham = ham.replace("'", "")
    kelime = ham
    if not kelime:
        return []
    harf_fonemleri = _kelime_fonemleri(kelime)
    vurgu = None if kelime in VURGUSUZ else _vurgulu_unlu(kelime, kok_uzunlugu)
    if vurgu is not None and len([h for h in kelime if h in UNLULER]) < 2 and kelime not in VURGU_ISTISNA:
        vurgu_yeri = 0  # tek heceli kelime: vurgu işareti başa
    elif vurgu is not None:
        # Türkçe hecede en fazla bir başlangıç ünsüzü var: vurgu, ünlüden önceki ünsüzden başlar
        vurgu_yeri = vurgu - 1 if vurgu > 0 and kelime[vurgu - 1] not in UNLULER else vurgu
        if not harf_fonemleri[vurgu_yeri]:  # sessiz ğ: vurgu doğrudan ünlüde başlar
            vurgu_yeri = vurgu
    fonemler: List[str] = []
    for i, f in enumerate(harf_fonemleri):
        if vurgu is not None and i == vurgu_yeri:
            fonemler.append("ˈ")
        for p in f:
            fonemler.extend(list(p) if p.endswith("ː") else [p])
    return fonemler


def fonemlere_cevir(metin: str) -> List[str]:
    """Normalize edilmiş Türkçe metni IPA fonem listesine çevirir (boşluk ve noktalama dahil)."""
    metin = kucult(metin).replace("’", "'").replace("`", "'")
    parcalar = re.findall(r"[a-zçğıöşüâîûêqwx']+|[,.?!;:…\-–—()]", metin)
    fonemler: List[str] = []
    for p in parcalar:
        if p in NOKTALAMA:
            if fonemler and fonemler[-1] == " ":
                fonemler.pop()
            if fonemler and fonemler[-1] in NOKTALAMA.values():
                continue
            fonemler.append(NOKTALAMA[p])
            fonemler.append(" ")
            continue
        ipa = _kelime_ipa(p.strip("'"))
        if ipa:
            fonemler.extend(ipa)
            fonemler.append(" ")
    while fonemler and fonemler[-1] == " ":
        fonemler.pop()
    return fonemler


def fonem_idleri(fonemler: List[str], fonem_id_haritasi: Dict[str, List[int]]) -> List[int]:
    """Piper biçiminde id dizisi: ^ _ f1 _ f2 _ ... $ (her fonemden sonra dolgu)."""
    dolgu = fonem_id_haritasi["_"]
    idler = list(fonem_id_haritasi["^"]) + list(dolgu)
    for f in fonemler:
        if f not in fonem_id_haritasi:
            raise KeyError(f"Bilinmeyen fonem: {f!r}")
        idler += list(fonem_id_haritasi[f]) + list(dolgu)
    return idler + list(fonem_id_haritasi["$"])
'''

os.makedirs(f"{ARAC}/tts_tr", exist_ok=True)
open(f"{ARAC}/tts_tr/g2p.py", "w").write(G2P_KODU)
open(f"{ARAC}/tts_tr/__init__.py", "w").write("from .g2p import fonemlere_cevir, fonem_idleri\n")
print("G2P hazır")

In [ ]:
# --- Veri: metadata.csv'yi bul, Piper biçimine (dataset.jsonl + önbellek) çevir ---
def veri_bul():
    # Kaggle girdiyi farklı klasörlere bağlayabiliyor; wavs/ klasörü yanında olan herhangi bir metadata.csv işimizi görür
    for kok, dizinler, dosyalar in os.walk(GIRDI, followlinks=True):
        if "metadata.csv" in dosyalar and os.path.isdir(f"{kok}/wavs"):
            return f"{kok}/metadata.csv"
    import zipfile
    for kok, _, dosyalar in os.walk(GIRDI, followlinks=True):
        for d in dosyalar:
            if d.endswith(".zip") and any(n.endswith("metadata.csv") for n in zipfile.ZipFile(f"{kok}/{d}").namelist()):
                shutil.unpack_archive(f"{kok}/{d}", f"{ARAC}/veri")
                return next(iter(glob.glob(f"{ARAC}/veri/**/metadata.csv", recursive=True)), None)
meta = veri_bul()
if not meta:
    print("Girdi klasöründekiler:")
    for kok, dizinler, dosyalar in os.walk(GIRDI, followlinks=True):
        if kok.count(os.sep) - GIRDI.count(os.sep) > 3: continue
        print(" ", kok, dosyalar[:5])
assert meta, "metadata.csv bulunamadı: sağ panelden Add Input → Your Work ile veri üretim notebook'unu ekle"
meta = [meta]
META = meta[0]
print("Veri:", META, "|", sum(1 for _ in open(META)), "cümle")

BASLANGIC = f"{ARAC}/lj-med_1000.ckpt"
if not os.path.exists(BASLANGIC):
    sh(f"curl -sL -o {BASLANGIC} https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_US/ljspeech/medium/lj-med_1000.ckpt")
    sh(f"curl -sL -o {ARAC}/lj_config.json https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_US/ljspeech/medium/config.json")

ON_ISLEME = r"""
import json, sys
from pathlib import Path
sys.path.insert(0, sys.argv[1]); sys.path.insert(0, sys.argv[2])
from tts_tr import fonemlere_cevir, fonem_idleri
from piper_train.norm_audio import cache_norm_audio, make_silence_detector
meta, lj_config, cikis = Path(sys.argv[3]), Path(sys.argv[4]), Path(sys.argv[5])
cikis.mkdir(parents=True, exist_ok=True); (cikis / "cache").mkdir(exist_ok=True)
config = json.load(open(lj_config))
config.update({"dataset": "emre_tr", "language": {"code": "tr_TR"}, "espeak": {"voice": "tr"},
               "phoneme_type": "tts_tr_g2p", "num_speakers": 1, "speaker_id_map": {}})
config["num_symbols"] = config.get("num_symbols", 256)
json.dump(config, open(cikis / "config.json", "w"), ensure_ascii=False, indent=2)
harita = config["phoneme_id_map"]
detector = make_silence_detector()
n = atlanan = 0
with open(cikis / "dataset.jsonl", "w") as f:
    for satir in open(meta):
        k, metin = satir.rstrip("\n").split("|", 1)
        try:
            idler = fonem_idleri(fonemlere_cevir(metin), harita)
        except KeyError as e:
            atlanan += 1; continue
        wav = meta.parent / "wavs" / f"{k}.wav"
        norm, spec = cache_norm_audio(wav, cikis / "cache", detector, config["audio"]["sample_rate"])
        f.write(json.dumps({"text": metin, "phoneme_ids": idler, "audio_norm_path": str(norm),
                            "audio_spec_path": str(spec)}, ensure_ascii=False) + "\n")
        n += 1
print(f"Hazır: {n} cümle, atlanan: {atlanan}")
"""
open(f"{ARAC}/on_isleme.py", "w").write(ON_ISLEME)
VERI = f"{ARAC}/piper_veri"
print(sh(f"{PY} {ARAC}/on_isleme.py {ARAC} {PIPER_PY} {META} {ARAC}/lj_config.json {VERI}"))
print(open(f"{VERI}/dataset.jsonl").readline()[:300])

In [ ]:
# --- Eğitim ---
# Önceki bir eğitimin checkpoint'i girdi olarak eklendiyse oradan, yoksa LJSpeech'ten devam eder.
onceki = sorted(f"{k}/emre_tr_son.ckpt" for k, _, d in os.walk(GIRDI, followlinks=True) if "emre_tr_son.ckpt" in d)
if not onceki: print("UYARI: girdide emre_tr_son.ckpt yok, eğitim LJSpeech checkpoint'inden SIFIRDAN başlıyor. Devam için önceki eğitimin çıktısını Add Input ile ekle.")
BASLA = onceki[0] if onceki else BASLANGIC
if onceki:
    # Checkpoint, önceki turun Timer sayacını (9.5 saat) da taşıyor; silmezsek eğitim ilk adımda "süre doldu" diye biter
    BASLA_TEMIZ = f"{ARAC}/devam.ckpt"
    subprocess.run([PY, "-c", "import sys, torch; c = torch.load(sys.argv[1], map_location='cpu'); c.pop('callbacks', None); torch.save(c, sys.argv[2])",
                    BASLA, BASLA_TEMIZ], check=True)
    BASLA = BASLA_TEMIZ
print("Başlangıç checkpoint'i:", onceki[0] if onceki else BASLANGIC)
for d in glob.glob(f"{VERI}/lightning_logs"):
    shutil.rmtree(d)
komut = [PY, "-m", "piper_train", "--dataset-dir", VERI, "--accelerator", "gpu" if "GPU" in sh(f"{PY} -c 'import torch; print(\"GPU\" if torch.cuda.is_available() else \"CPU\")'") else "cpu",
         "--devices", "1", "--batch-size", str(BATCH_SIZE), "--validation-split", "0", "--num-test-examples", "0",
         "--max_epochs", "100000", "--max_time", EGITIM_SURESI, "--resume_from_checkpoint", BASLA,
         "--checkpoint-epochs", str(KAYIT_ARALIGI), "--precision", "32", "--quality", "medium"]
komut += os.environ.get("EK_EGITIM_ARGUMANLARI", "").split()
p = subprocess.Popen(komut, cwd=PIPER_PY, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                     env={**os.environ, "PYTHONPATH": PIPER_PY})
son_satir = ""
for satir in p.stdout:
    if "Epoch" in satir and "loss" in satir:
        son_satir = satir
    elif any(s in satir for s in ("Error", "Traceback", "error:", "Restoring", "max_time", "Trainer.fit")):
        print(satir, end="")
    if "Epoch" in satir and satir.count("100%"):
        print(satir.strip()[-160:])
p.wait()
print("Eğitim bitti, çıkış kodu:", p.returncode, "| son:", son_satir.strip()[-200:])
if p.returncode != 0 or not glob.glob(f"{VERI}/lightning_logs/version_*/checkpoints/*.ckpt"):
    raise RuntimeError("Eğitim checkpoint üretmeden bitti; yukarıdaki satırları bana gönder")

In [ ]:
# --- Dışa aktar: ONNX modeli + config, devam için son checkpoint, örnek sesler ---
ckptler = sorted(glob.glob(f"{VERI}/lightning_logs/version_*/checkpoints/*.ckpt"), key=os.path.getmtime)
assert ckptler, "Checkpoint oluşmadı; eğitim çıktısına bak"
SON = ckptler[-1]; print("Son checkpoint:", os.path.basename(SON))
shutil.copy(SON, f"{CALISMA}/emre_tr_son.ckpt")
MODEL = f"{CALISMA}/tr_TR-emre-medium.onnx"
sh(f"cd {PIPER_PY} && {PY} -m piper_train.export_onnx {SON} {MODEL}")
config = json.load(open(f"{VERI}/config.json"))
json.dump(config, open(MODEL + ".json", "w"), ensure_ascii=False, indent=2)

ORNEK = r"""
import json, sys, numpy as np, onnxruntime as ort, soundfile as sf
sys.path.insert(0, sys.argv[1])
from tts_tr import fonemlere_cevir, fonem_idleri
model, cikis = sys.argv[2], sys.argv[3]
ek = sys.argv[5] if len(sys.argv) > 5 else ""
config = json.load(open(model + ".json"))
s = ort.InferenceSession(model)
for i, metin in enumerate(json.load(open(sys.argv[4], encoding="utf-8")), 1):
    idler = np.array([fonem_idleri(fonemlere_cevir(metin), config["phoneme_id_map"])], dtype=np.int64)
    ses = s.run(None, {"input": idler, "input_lengths": np.array([idler.shape[1]], dtype=np.int64),
                       "scales": np.array([0.667, 1.0, 0.8], dtype=np.float32)})[0].squeeze()
    sf.write(f"{cikis}/ornek_{i:02d}{ek}.wav", ses / max(abs(ses).max(), 1e-6) * 0.9, config["audio"]["sample_rate"])
    print(f"ornek_{i:02d}{ek}.wav", metin)
"""
open(f"{ARAC}/ornek.py", "w").write(ORNEK)
json.dump(TEST_CUMLELERI, open(f"{ARAC}/test_cumleleri.json", "w", encoding="utf-8"), ensure_ascii=False)
print(sh(f"{PY} {ARAC}/ornek.py {ARAC} {MODEL} {CALISMA} {ARAC}/test_cumleleri.json"))
# Telefon ve paylaşım için 8 bitlik küçük kopya (~19 MB); kalite farkını _int8 örnekleriyle karşılaştır
MODEL8 = f"{CALISMA}/tr_TR-emre-medium-int8.onnx"
subprocess.run([PY, "-c", "import sys; from onnxruntime.quantization import quantize_dynamic, QuantType; quantize_dynamic(sys.argv[1], sys.argv[2], weight_type=QuantType.QUInt8)", MODEL, MODEL8], check=True, capture_output=True)
shutil.copy(MODEL + ".json", MODEL8 + ".json")
print(sh(f"{PY} {ARAC}/ornek.py {ARAC} {MODEL8} {CALISMA} {ARAC}/test_cumleleri.json _int8"))
from IPython.display import Audio, display
for i, m in enumerate(TEST_CUMLELERI, 1):
    print(m); display(Audio(f"{CALISMA}/ornek_{i:02d}.wav"))
print("Model boyutu:", round(os.path.getsize(MODEL) / 1e6, 1), "MB | int8:", round(os.path.getsize(MODEL8) / 1e6, 1), "MB")